# Suite P3PRT — Printing Python 3 modules from the store

`Programs.Python3.Print` prints a module as the store holds it: owned by `Programs.Framework.TheWorkspace` by its path,
its statements owned through `Programs.Framework.HasParent`, keyed by property and index, and its expressions
mbse-expressions' terms, which `Python3.Text.ToText` writes. The layout is the 0.3 printer's: four spaces a level,
`elif` for an `else` that is one `if`, PEP 8's blank lines around definitions, comments kept with the definition they
lead, trailing comments on the line they end. Where a statement's place takes a bare tuple, `yield` or named
expression, it is written bare.

Each case builds its module in a transaction of its own with the builders, its expressions read from Python source by
`Python3.Text.FromText`, and checks the printed text, which CPython's `ast` must parse. The case setup is below.

In [ ]:
import ast

from mbse.Expressions import Framework
from mbse.Expressions.IEEE754 import Expressions as IEEE754
from mbse.Expressions.ISOIEC11404 import Expressions as ISOIEC11404
from mbse.Expressions.Python import Patterns as PythonPatterns, Python3, Types as PythonTypes
from mbse.Expressions.Python.Python3.Text import FromText as X
from mbse.Programs import Python3 as Py
from mbse.Programs.Framework import Model
from mbse.Programs.Python3 import Print, Statements
from mbse.Schemas.Framework import Builders as B
from support import raises

INSTALLS = (Framework.install, ISOIEC11404.install, IEEE754.install, Python3.install, PythonTypes.install,
            PythonPatterns.install, Model.install, Py.install)
store = B.Store()
for install in INSTALLS:
    store.transaction(install)
R = store.registered


def under(prop, *children):
    """The entries that make `children` a parent's, under its property `prop`, indexed in order."""
    return [lambda e, c=c, i=i: e.child(c).property(prop).index(i) for i, c in enumerate(children)]


def build(path, kinds, specs):
    """The module at `path`, built in a transaction of its own: `kinds` gives each symbol's statement kind, `specs`
    the builders' specs over the symbols, `_.m` the module."""
    store.transaction(lambda t: t.uid(path).mutations([lambda mu: mu.upsert(lambda u: u.exists([lambda p: p
        .symbols({"ws": R("Programs.Framework.Workspace"), "m": R("Programs.Python3.Module"),
                  **{symbol: R(f"Programs.Python3.{kind}") for symbol, kind in kinds.items()}})
        .spec([lambda _: _.ws.files({path: lambda e: e.file(_.m)}), *specs])]))]))


def printed(path, kinds, specs, parses=True):
    """The text of the module built at `path`, which CPython's `ast` parses, where `parses`."""
    build(path, kinds, specs)
    text = Print.print_module(store, path)
    if parses:
        ast.parse(text)
    return text

<a id="P3PRT-01"></a>
## P3PRT-01 · Every simple statement kind prints alone, on its line

An expression statement, assignments plain, augmented and annotated, `assert`, `pass`, `break`, `continue`, `del`,
`return`, `raise`, imports with their aliases, relative and `*`, `global`, `nonlocal`, a type alias with its type
parameters, and a comment, whose text follows its `#`. `lazy` imports (Python 3.15) print too.

In [ ]:
kinds = {"e": "Expr", "a": "Assign", "g": "AugAssign", "n": "AnnAssign", "n2": "AnnAssign", "s": "Assert",
         "s2": "Assert", "p": "Pass", "b": "Break", "c": "Continue", "d": "Delete", "r": "Return", "r2": "Return",
         "x": "Raise", "x2": "Raise", "x3": "Raise", "i": "Import", "f": "ImportFrom", "f2": "ImportFrom",
         "f3": "ImportFrom", "gl": "Global", "nl": "Nonlocal", "t": "TypeAlias", "t2": "TypeAlias", "k": "Comment"}
text = printed("simple.py", kinds, [
    lambda _: _.m.body(under("body", *(getattr(_, symbol) for symbol in kinds))),
    lambda _: _.e.value(X("f(x)")),
    lambda _: _.a.targets([X("x")]).value(X("1")),
    lambda _: _.g.target(X("x")).op("<<").value(X("2")),
    lambda _: _.n.target(X("x")).annotation(X("int")).value(X("1")),
    lambda _: _.n2.target(X("self.y")).annotation(X("list[int]")),
    lambda _: _.s.test(X("x > 0")),
    lambda _: _.s2.test(X("x")).msg(X("'m'")),
    lambda _: _.d.targets([X("a"), X("b[0]"), X("c.d")]),
    lambda _: _.r2.value(X("x")),
    lambda _: _.x2.exc(X("E")),
    lambda _: _.x3.exc(X("E('m')")).cause(X("error")),
    lambda _: _.i.names([lambda a: a.name("os"), lambda a: a.name("numpy.linalg").asname("la")]),
    lambda _: _.f.level(1).names([lambda a: a.name("a")]),
    lambda _: _.f2.level(2).module("pkg.sub").names([lambda a: a.name("b").asname("c"), lambda a: a.name("d")]),
    lambda _: _.f3.level(0).module("os").names([lambda a: a.name("*")]),
    lambda _: _.gl.names(["a", "b"]),
    lambda _: _.nl.names(["c"]),
    lambda _: _.t.name(X("Pair")).type_params([lambda u: u.typevar(lambda v: v.name("T"))]).value(X("tuple[T, T]")),
    lambda _: _.t2.name(X("Alias")).value(X("int | None")),
    lambda _: _.k.text(" a note"),
])
assert text == """\
f(x)
x = 1
x <<= 2
x: int = 1
self.y: list[int]
assert x > 0
assert x, 'm'
pass
break
continue
del a, b[0], c.d
return
return x
raise
raise E
raise E('m') from error
import os, numpy.linalg as la
from . import a
from ..pkg.sub import b as c, d
from os import *
global a, b
nonlocal c
type Pair[T] = tuple[T, T]
type Alias = int | None
# a note
"""
lazy = printed("lazy.py", {"i": "Import", "f": "ImportFrom"}, [
    lambda _: _.m.body(under("body", _.i, _.f)),
    lambda _: _.i.is_lazy(True).names([lambda a: a.name("json")]),
    lambda _: _.f.is_lazy(True).level(0).module("os").names([lambda a: a.name("path")]),
], parses=False)  # Python 3.15's
assert lazy == "lazy import json\nlazy from os import path\n"

<a id="P3PRT-02"></a>
## P3PRT-02 · Every compound statement kind prints, with its clauses and its parts

`if`, `while` and `for` with their `else`, `async for` and `async with`, `try` with its `except` clauses (a type, a
tuple, a name, none), `else` and `finally`, `try` with `except*`, `with` and its items, `match` with its cases'
patterns and guards, and function and class definitions with their decorators, type parameters, parameters of every
kind, return type, bases and keywords.

In [ ]:
loops = printed("loops.py", {"i": "If", "w": "While", "f": "For", "p1": "Pass", "p2": "Pass", "p3": "Pass",
                             "p4": "Pass", "p5": "Pass", "p6": "Pass"}, [
    lambda _: _.m.body(under("body", _.i, _.w, _.f)),
    lambda _: _.i.test(X("a")).children(under("body", _.p1) + under("orelse", _.p2)),
    lambda _: _.w.test(X("a < b")).children(under("body", _.p3) + under("orelse", _.p4)),
    lambda _: _.f.target(X("x")).iter(X("range(3)")).children(under("body", _.p5) + under("orelse", _.p6)),
])
assert loops == """\
if a:
    pass
else:
    pass
while a < b:
    pass
else:
    pass
for x in range(3):
    pass
else:
    pass
"""
asynchronous = printed("async.py", {"f": "AsyncFunctionDef", "l": "AsyncFor", "w": "AsyncWith", "e": "Expr",
                                    "p": "Pass"}, [
    lambda _: _.m.body(under("body", _.f)),
    lambda _: _.f.name("main").children(under("body", _.l)),
    lambda _: _.l.target(X("x")).iter(X("stream()")).children(under("body", _.w)),
    lambda _: _.w.items([lambda w: w.context_expr(X("lock"))]).children(under("body", _.e)),
    lambda _: _.e.value(X("await handle(x)")),
])
assert asynchronous == """\
async def main():
    async for x in stream():
        async with lock:
            await handle(x)
"""
tries = printed("try.py", {"t": "Try", "h1": "ExceptHandler", "h2": "ExceptHandler", "h3": "ExceptHandler",
                           "s": "TryStar", "h4": "ExceptHandler", "p1": "Pass", "p2": "Pass", "p3": "Pass", "p4": "Pass",
                           "p5": "Pass", "p6": "Pass", "p7": "Pass", "p8": "Pass"}, [
    lambda _: _.m.body(under("body", _.t, _.s)),
    lambda _: _.t.children(under("body", _.p1) + under("handlers", _.h1, _.h2, _.h3) + under("orelse", _.p5)
                           + under("finalbody", _.p6)),
    lambda _: _.h1.type(X("ValueError")).name("e").children(under("body", _.p2)),
    lambda _: _.h2.type(X("(KeyError, IndexError)")).children(under("body", _.p3)),
    lambda _: _.h3.children(under("body", _.p4)),
    lambda _: _.s.children(under("body", _.p7) + under("handlers", _.h4)),
    lambda _: _.h4.type(X("OSError")).name("group").children(under("body", _.p8)),
])
assert tries == """\
try:
    pass
except ValueError as e:
    pass
except (KeyError, IndexError):
    pass
except:
    pass
else:
    pass
finally:
    pass
try:
    pass
except* OSError as group:
    pass
"""
withs = printed("with.py", {"w": "With", "p": "Pass"}, [
    lambda _: _.m.body(under("body", _.w)),
    lambda _: _.w.items([lambda w: w.context_expr(X("open(path)")).optional_vars(X("f")),
                         lambda w: w.context_expr(X("a")).optional_vars(X("(b, c)")),
                         lambda w: w.context_expr(X("lock"))]).children(under("body", _.p)),
])
assert withs == "with open(path) as f, a as (b, c), lock:\n    pass\n"

In [ ]:
def value(term):
    return lambda u: u.matchvalue(lambda v: v.value(term))


def capture(name):
    return lambda u: u.matchas(lambda m: m.name(name))


cases = {f"c{i}": "MatchCase" for i in range(8)} | {f"p{i}": "Pass" for i in range(8)}
match = printed("match.py", {"s": "Match", **cases}, [
    lambda _: _.m.body(under("body", _.s)),
    lambda _: _.s.subject(X("command")).children(under("cases", *(getattr(_, f"c{i}") for i in range(8)))),
    *(lambda _, i=i: getattr(_, f"c{i}").children(under("body", getattr(_, f"p{i}"))) for i in range(8)),
    lambda _: _.c0.pattern(lambda u: u.matchsequence(lambda s: s.delimiters("[]").patterns([capture("action")]))),
    lambda _: _.c1.pattern(lambda u: u.matchclass(lambda c: c.cls(X("Point")).patterns([value(X("0"))])
                                                  .kwd_attrs(["y"]).kwd_patterns([capture("y")]))).guard(X("y > 0")),
    lambda _: _.c2.pattern(lambda u: u.matchmapping(lambda m: m.keys([X("'k'")]).patterns([capture("v")])
                                                    .rest("rest"))),
    lambda _: _.c3.pattern(lambda u: u.matchas(lambda m: m.name("n").pattern(lambda p: p.matchor(
        lambda o: o.patterns([value(X("1")), value(X("2"))]))))),
    lambda _: _.c4.pattern(lambda u: u.matchsequence(lambda s: s.patterns([capture("a"), lambda p: p.matchstar(
        lambda m: m.name("others"))]))),
    lambda _: _.c5.pattern(lambda u: u.matchsingleton(lambda m: m.value("None"))),
    lambda _: _.c6.pattern(lambda u: u.matchsequence(lambda s: s.delimiters("()").patterns([capture("x")]))),
    lambda _: _.c7.pattern(lambda u: u.matchas(lambda m: m)),
])
assert match == """\
match command:
    case [action]:
        pass
    case Point(0, y=y) if y > 0:
        pass
    case {'k': v, **rest}:
        pass
    case 1 | 2 as n:
        pass
    case a, *others:
        pass
    case None:
        pass
    case (x,):
        pass
    case _:
        pass
"""

In [ ]:
definitions = printed("definitions.py", {"f": "FunctionDef", "c": "ClassDef", "p1": "Pass", "p2": "Pass"}, [
    lambda _: _.m.body(under("body", _.f, _.c)),
    lambda _: _.f.decorator_list([X("cache"), X("route('/a')")]).name("f").type_params([
        lambda u: u.typevar(lambda t: t.name("T").bound(X("int")).default_value(X("bool"))),
        lambda u: u.typevartuple(lambda t: t.name("Ts")),
        lambda u: u.paramspec(lambda t: t.name("P"))]).args(lambda a: a
        .posonlyargs([lambda g: g.arg("a")])
        .args([lambda g: g.arg("b").annotation(X("T")), lambda g: g.arg("c").default_value(X("1")),
               lambda g: g.arg("d").annotation(X("int")).default_value(X("2"))])
        .vararg(lambda g: g.arg("rest").annotation(X("*Ts")))
        .kwonlyargs([lambda g: g.arg("e"), lambda g: g.arg("k").default_value(X("None"))])
        .kwarg(lambda g: g.arg("options"))).returns(X("T | None")).children(under("body", _.p1)),
    lambda _: _.c.decorator_list([X("dataclass")]).name("C").type_params([lambda u: u.typevar(lambda t: t.name("T"))])
        .bases([X("Base[T]"), X("Mixin")]).keywords([lambda k: k.arg("metaclass").value(X("Meta")),
                                                     lambda k: k.value(X("options"))]).children(under("body", _.p2)),
])
assert definitions == """\
@cache
@route('/a')
def f[T: int = bool, *Ts, **P](a, /, b: T, c=1, d: int = 2, *rest: *Ts, e, k=None, **options) -> T | None:
    pass


@dataclass
class C[T](Base[T], Mixin, metaclass=Meta, **options):
    pass
"""
bare = printed("bare.py", {"f": "FunctionDef", "g": "FunctionDef", "c": "ClassDef", "p1": "Pass", "p2": "Pass",
                           "p3": "Pass"}, [
    lambda _: _.m.body(under("body", _.f, _.g, _.c)),
    lambda _: _.f.name("f").children(under("body", _.p1)),
    lambda _: _.g.name("g").args(lambda a: a.kwonlyargs([lambda g: g.arg("k")])).children(under("body", _.p2)),
    lambda _: _.c.name("C").children(under("body", _.p3)),
])
assert bare == "def f():\n    pass\n\n\ndef g(*, k):\n    pass\n\n\nclass C:\n    pass\n"

<a id="P3PRT-03"></a>
## P3PRT-03 · Layout: blank lines around definitions, and comments

Two blank lines around a top-level definition and one around a nested one; comments directly before a definition stay
with it, the blank lines before them; a trailing comment ends the line of what it follows, and the first of a body
ends its header's line.

In [ ]:
layout = printed("layout.py", {"i": "Import", "k1": "Comment", "f": "FunctionDef", "c": "ClassDef", "doc": "Expr",
                               "m1": "FunctionDef", "k2": "Comment", "k3": "Comment", "m2": "FunctionDef",
                               "pc": "Pass", "k4": "Comment", "a": "Assign", "p1": "Pass", "p2": "Pass", "p3": "Pass",
                               "k5": "Comment"}, [
    lambda _: _.m.body(under("body", _.i, _.k1, _.f, _.c, _.k4, _.a)),
    lambda _: _.i.names([lambda a: a.name("os")]),
    lambda _: _.k1.text(" lead"),
    lambda _: _.f.name("f").children(under("body", _.k5, _.p1)),
    lambda _: _.k5.text(" on the header").trailing(True),
    lambda _: _.c.decorator_list([X("d")]).name("C").children(under("body", _.doc, _.m1, _.k2, _.k3, _.m2, _.pc)),
    lambda _: _.doc.value(X("'doc'")),
    lambda _: _.m1.name("m").args(lambda a: a.args([lambda g: g.arg("self")])).children(under("body", _.p2)),
    lambda _: _.k2.text(" first"),
    lambda _: _.k3.text(" second"),
    lambda _: _.m2.name("n").args(lambda a: a.args([lambda g: g.arg("self")])).children(under("body", _.p3)),
    lambda _: _.k4.text(" trailing").trailing(True),
    lambda _: _.a.targets([X("x")]).value(X("1")),
])
assert layout == """\
import os


# lead
def f():  # on the header
    pass


@d
class C:
    'doc'

    def m(self):
        pass

    # first
    # second
    def n(self):
        pass

    pass  # trailing


x = 1
"""

<a id="P3PRT-04"></a>
## P3PRT-04 · `elif` for an `else` that is one `if`

An `if` whose `orelse` is one `If` prints as `elif`, in a chain; an `orelse` holding anything else beside it, a comment
among them, prints as `else` with the `if` nested.

In [ ]:
chain = printed("chain.py", {"i": "If", "j": "If", "k": "If", "t": "Comment", "p1": "Pass", "p2": "Pass", "p3": "Pass",
                             "p4": "Pass"}, [
    lambda _: _.m.body(under("body", _.i)),
    lambda _: _.i.test(X("a")).children(under("body", _.p1) + under("orelse", _.j)),
    lambda _: _.j.test(X("b")).children(under("body", _.t, _.p2) + under("orelse", _.k)),
    lambda _: _.t.text(" c").trailing(True),
    lambda _: _.k.test(X("c")).children(under("body", _.p3) + under("orelse", _.p4)),
])
assert chain == "if a:\n    pass\nelif b:  # c\n    pass\nelif c:\n    pass\nelse:\n    pass\n"
nested = printed("nested.py", {"i": "If", "j": "If", "k": "Comment", "p1": "Pass", "p2": "Pass"}, [
    lambda _: _.m.body(under("body", _.i)),
    lambda _: _.i.test(X("a")).children(under("body", _.p1) + under("orelse", _.k, _.j)),
    lambda _: _.k.text(" keeps else"),
    lambda _: _.j.test(X("b")).children(under("body", _.p2)),
])
assert nested == "if a:\n    pass\nelse:\n    # keeps else\n    if b:\n        pass\n"

<a id="P3PRT-05"></a>
## P3PRT-05 · Precedence where a statement's place takes more than an expression

`ToText` writes a tuple in parentheses and a named expression or a `yield` in their own; where a statement's place
takes them bare, they print bare: a tuple as an expression statement, an assignment's targets and value, a `return`'s
value, a `for`'s target and iterable and a `match`'s subject; a `yield` as an expression statement and an assignment's
value; a named expression as the test of an `if`, an `elif` and a `while`, a guard, a decorator and a subject. Elsewhere
they keep their parentheses, and an `except` clause's tuple keeps its own.

In [ ]:
kinds = {"e1": "Expr", "e2": "Expr", "e3": "Expr", "e4": "Expr", "e5": "Expr", "a1": "Assign", "a2": "Assign", "a3": "Assign",
         "a4": "Assign", "a5": "Assign", "g": "AugAssign", "n": "AnnAssign", "f": "FunctionDef", "r1": "Return",
         "r2": "Return", "r3": "Return", "o": "For", "i": "If", "j": "If", "w": "While", "s": "Match", "c": "MatchCase",
         "t": "Try", "h": "ExceptHandler", "x": "Assert", "d": "Delete", "w2": "With",
         **{f"p{i}": "Pass" for i in range(9)}}
statements = printed("precedence.py", kinds, [
    lambda _: _.m.body(under("body", _.e1, _.e2, _.e3, _.e4, _.e5, _.a1, _.a2, _.a3, _.a4, _.a5, _.g, _.n, _.f, _.o, _.i,
                             _.w, _.s, _.t, _.x, _.d, _.w2)),
    lambda _: _.e1.value(X("a, b")),
    lambda _: _.e2.value(X("yield a")),
    lambda _: _.e3.value(X("(x := 1)")),
    lambda _: _.e4.value(X("(a + b) * c")),
    lambda _: _.e5.value(X("()")),
    lambda _: _.a1.targets([X("a, b")]).value(X("b, a")),
    lambda _: _.a2.targets([X("x"), X("(y,)")]).value(X("yield")),
    lambda _: _.a3.targets([X("y")]).value(X("(x := 1)")),
    lambda _: _.a4.targets([X("f")]).value(X("lambda x: x")),
    lambda _: _.a5.targets([X("z")]).value(X("yield from g()")),
    lambda _: _.g.target(X("total")).op("+").value(X("yield")),
    lambda _: _.n.target(X("pair")).annotation(X("tuple[int, int]")).value(X("1, 2")),
    lambda _: _.f.decorator_list([X("(d := make())")]).name("f").children(under("body", _.r1, _.r2, _.r3)),
    lambda _: _.r1.value(X("a, b")),
    lambda _: _.r2.value(X("(x := 1)")),
    lambda _: _.r3.value(X("(yield)")),
    lambda _: _.o.target(X("k, v")).iter(X("a, b")).children(under("body", _.p0)),
    lambda _: _.i.test(X("(n := f())")).children(under("body", _.p1) + under("orelse", _.j)),
    lambda _: _.j.test(X("(m := g()) > 0")).children(under("body", _.p2)),
    lambda _: _.w.test(X("(line := read())")).children(under("body", _.p3)),
    lambda _: _.s.subject(X("a, (b := c)")).children(under("cases", _.c)),
    lambda _: _.c.pattern(lambda u: u.matchas(lambda m: m.name("x"))).guard(X("(y := x)"))
        .children(under("body", _.p4)),
    lambda _: _.t.children(under("body", _.p5) + under("handlers", _.h)),
    lambda _: _.h.type(X("(A, B)")).children(under("body", _.p6)),
    lambda _: _.x.test(X("(a, b)")).msg(X("(x := 1)")),
    lambda _: _.d.targets([X("(a, b)"), X("c")]),
    lambda _: _.w2.items([lambda w: w.context_expr(X("(yield)")).optional_vars(X("v"))]).children(under("body", _.p7)),
])
assert statements == """\
a, b
yield a
(x := 1)
(a + b) * c
()
a, b = b, a
x = y, = yield
y = (x := 1)
f = lambda x: x
z = yield from g()
total += yield
pair: tuple[int, int] = 1, 2


@d := make()
def f():
    return a, b
    return (x := 1)
    return (yield)


for k, v in a, b:
    pass
if n := f():
    pass
elif (m := g()) > 0:
    pass
while line := read():
    pass
match a, (b := c):
    case x if y := x:
        pass
try:
    pass
except (A, B):
    pass
assert (a, b), (x := 1)
del (a, b), c
with (yield) as v:
    pass
"""

<a id="P3PRT-06"></a>
## P3PRT-06 · Reading the store: transactions, visitors, order, and every kind

`print_module` reads within the transaction open on the store, or within one of its own, and refuses a path the
workspace holds no file at; `print_file` prints a module's visitor and `print_statement` a statement's, each within
the transaction that made it, an `except*` clause alone by its `try`. Children print in the order of their property's
index, whatever the order their entries were written in. The printer prints every statement kind.

In [ ]:
build("order.py", {"a": "Assign", "b": "Assign", "s": "TryStar", "h": "ExceptHandler", "p1": "Pass", "p2": "Pass"}, [
    lambda _: _.m.body([lambda e: e.child(_.b).property("body").index(1), lambda e: e.child(_.s).property("body").index(2),
                        lambda e: e.child(_.a).property("body").index(0)]),
    lambda _: _.a.targets([X("x")]).value(X("1")),
    lambda _: _.b.targets([X("y")]).value(X("2")),
    lambda _: _.s.children([lambda e: e.child(_.h).property("handlers").index(0),
                            lambda e: e.child(_.p1).property("body").index(0)]),
    lambda _: _.h.type(X("E")).children(under("body", _.p2)),
])
expected = "x = 1\ny = 2\ntry:\n    pass\nexcept* E:\n    pass\n"
assert Print.print_module(store, "order.py") == expected  # in a transaction of its own
with store.transaction():
    with raises(ValueError, match="transactions do not nest"):  # print_module opens its own: called with none open
        Print.print_module(store, "order.py")
    module = store.singleton(Model.WORKSPACE).files({"path": "order.py"})
    assert Print.print_file(module) == expected
    entries = sorted(module.body, key=lambda entry: entry.index.get())
    assert [Print.print_statement(entry.child) for entry in entries[:2]] == ["x = 1", "y = 2"]
    (handler,) = [entry.child for entry in entries[2].child.children if entry.property.get() == "handlers"]
    assert Print.print_statement(handler) == "except* E:\n    pass"
    with raises(TypeError, match="not a Programs.Python3.Module"):
        Print.print_file(entries[0].child)
    with raises(TypeError, match="no printer for the statement kind 'Programs.Python3.Module'"):
        Print.print_statement(module)
with raises(LookupError, match="'nope.py'"):
    Print.print_module(store, "nope.py")
build("empty.py", {}, [])
assert Print.print_module(store, "empty.py") == ""
assert sorted(Print.KINDS) == sorted(Statements.KINDS)